# Redrob Hackathon — Candidate Ranking Pipeline

Recruiter-style ranking of **Top 100** candidates for the *Founding Senior AI Engineer* role.

**Guiding philosophy:** *Eliminate only candidates who are extremely unlikely to be selected per the JD. Everything else is expressed as preferences through scoring.*

## Pipeline
```
100,000 Candidates
      │
      ▼  Stage 1: Direct Eliminations  (hard rejects)
Filtered Pool
      │
      ▼  Stage 2: Candidate Scoring     (weighted sub-scores)
Scored Candidates
      │
      ▼  Stage 3: Honeypot Detection    (reject / penalize)
Final Top 100
      │
      ▼  Reasoning Generation
submission.csv
```

## Compute constraints (from `submission_spec.md`)
- **≤ 5 min** wall-clock, **CPU only**, **≤ 16 GB RAM**, **no network during ranking**, **≤ 5 GB disk**.
- Pre-computation (model download, pip installs) is allowed **outside** the 5-min window.
- Output CSV columns: `candidate_id, rank, score, reasoning` (100 rows, `score` monotonically non-increasing).
- **Honeypot rate > 10% in top 100 = disqualification.**

> **Run order:** execute cells top-to-bottom. The first cell installs dependencies (run once, requires network). After that, the ranking itself uses no network."

## 0. Setup &amp; Configuration

The cell below installs dependencies (run once, needs network). `sentence-transformers` is optional — if it (or the model) is unavailable, the pipeline automatically falls back to **BM25 + TF-IDF** for semantic matching.

In [2]:
# Run ONCE (needs network). Safe to re-run; it is a no-op if already installed.
%pip install -q pandas numpy scikit-learn rank-bm25 sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

# ----------------------------- RUN CONFIG ---------------------------------
DATA_PATH = "candidates.jsonl"
OUTPUT_CSV = "submission.csv"
TOP_N = 100

# Semantic matching backend. If sentence-transformers / model unavailable,
# the pipeline degrades gracefully to BM25 + TF-IDF.
USE_DENSE_EMBEDDINGS = True
EMBED_MODEL = "all-MiniLM-L6-v2"   # small, CPU-friendly, ~80MB

# Reference "today" for recency calculations. The dataset is synthetic and
# dated in the future, so we anchor recency to the most recent last_active_date
# observed in the data (computed at load time). Override here if desired.
REFERENCE_DATE = None  # None -> auto (max last_active_date)

# ------------------------- AGGREGATION WEIGHTS ----------------------------
# Final score = weighted average of sub-scores, then multiplied by the
# CV/Robotics/Speech penalty multiplier (<=1.0). Weights sum to 1.0.
WEIGHTS = {
    "jd_alignment": 0.30,   # semantic + keyword fit to the JD
    "experience":   0.20,   # applied-AI years + total years (peaked)
    "behavioral":   0.15,   # redrob signals (availability / engagement)
    "production":   0.15,   # evidence of shipping production ML
    "location":     0.10,   # geography vs preferred hubs
    "like_to_have": 0.10,   # nice-to-have bonuses
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

print("Config loaded. Weights:", WEIGHTS)

Config loaded. Weights: {'jd_alignment': 0.3, 'experience': 0.2, 'behavioral': 0.15, 'production': 0.15, 'location': 0.1, 'like_to_have': 0.1}


In [4]:
# ============================ LEXICONS ====================================
# All matching is lowercase substring / token based (fast, no heavy regex).

# Preferred metro hubs (Stage 1 location filter + Stage 2 location score)
PREFERRED_PRIMARY = ["pune", "noida"]
PREFERRED_SECONDARY = ["delhi", "ncr", "gurgaon", "gurugram", "mumbai", "hyderabad"]
PREFERRED_LOCATIONS = PREFERRED_PRIMARY + PREFERRED_SECONDARY

# ----- AI/ML relevant SKILLS (exhaustive mapping over the 133 dataset skills
#       + JD terminology + synonyms). Matched against the skills[].name field.
RELEVANT_SKILLS = [
    "nlp", "natural language", "machine learning", "deep learning", "pytorch",
    "tensorflow", "keras", "jax", "scikit-learn", "sklearn", "xgboost", "lightgbm",
    "embeddings", "sentence transformers", "bert", "transformers", "hugging face",
    "huggingface", "llm", "llms", "large language", "fine-tuning", "fine tuning",
    "fine-tuning llms", "lora", "qlora", "peft", "rag", "langchain", "llamaindex",
    "prompt engineering", "information retrieval", "retrieval", "semantic search",
    "vector search", "learning to rank", "ranking", "recommendation systems",
    "recommendation system", "recommender", "bm25", "faiss", "pinecone", "weaviate",
    "qdrant", "milvus", "pgvector", "chroma", "elasticsearch", "opensearch",
    "neural networks", "neural network", "feature engineering", "mlops", "mlflow",
    "kubeflow", "bentoml", "weights & biases", "wandb", "statistical modeling",
    "data science", "diffusion models", "reinforcement learning", "cnn", "rnn",
    "lstm", "gan", "computer vision", "image classification", "object detection",
    "speech recognition", "asr", "tts", "yolo", "opencv", "haystack",
]

# ----- Relevant TITLE keywords (current OR past title)
RELEVANT_TITLE_KEYWORDS = [
    "ml engineer", "machine learning", "data scientist", "applied scientist",
    "applied ml", "ai engineer", "nlp engineer", "search engineer",
    "relevance engineer", "recommendation", "recommender", "research engineer",
    "deep learning", "ai/ml", "ai specialist", "ml scientist", "ranking engineer",
    "computer vision", "research scientist", "ml researcher",
]

# Titles that count as "tech" for the non-tech elimination whitelist
TECH_WHITELIST_TITLES = [
    "engineer", "developer", "scientist", "architect", "data analyst",
    "analytics engineer", "data engineer", "bi engineer", "research",
    "programmer", "sde", "swe", "devops", "sre", "platform", "ml", "ai",
    "cto", "vp engineering", "tech lead", "technical lead",
]

# Non-tech title categories (entire-career-non-tech elimination)
NON_TECH_TITLE_KEYWORDS = [
    "marketing", "sales", "accountant", "account executive", "hr ", "human resources",
    "recruiter", "talent acquisition", "operations manager", "content writer",
    "customer support", "customer success", "teacher", "finance", "legal",
    "business development", "brand", "seo specialist", "logistics",
    "mechanical engineer", "civil engineer", "electrical engineer",  # non-software eng
]

# ----- Career-description EVIDENCE of relevant work (Stage 1 + JD alignment)
CAREER_EVIDENCE_KEYWORDS = [
    "retrieval", "ranking", "re-rank", "rerank", "recommendation", "recommender",
    "relevance", "search", "embedding", "embeddings", "semantic", "vector",
    "evaluation", "ndcg", "mrr", "recall@", "precision@", "deployed", "production",
    "machine learning", "deep learning", "nlp", "language model", "llm",
    "fine-tune", "fine-tuning", "classifier", "model", "inference", "pipeline",
]

# ----- Consulting / services companies (consulting-only elimination)
CONSULTING_COMPANIES = [
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "hcl", "tech mahindra", "mindtree", "mphasis", "ltimindtree",
    "lti", "l&t infotech", "deloitte", "ey", "kpmg", "pwc", "ibm global",
    "dxc", "hexaware", "birlasoft", "coforge", "persistent systems",
]

# ----- Retrieval / Eval / LLM signal lexicons (JD alignment sub-signals)
RETRIEVAL_SIGNALS = [
    "retrieval", "search", "relevance", "ranking", "recommendation", "matching",
    "embedding", "semantic", "vector", "bm25", "learning to rank",
]
EVAL_SIGNALS = [
    "ndcg", "mrr", "map ", "mean average precision", "evaluation", "offline",
    "online", "benchmark", "a/b test", "ab test", "precision@", "recall@",
]
LLM_SIGNALS = [
    "prompt engineering", "fine-tun", "fine tun", "llm", "large language",
    "rag", "deployment", "gpt", "instruction tun", "peft", "lora",
]

# ----- Production evidence lexicon
PRODUCTION_KEYWORDS = [
    "production", "deployed", "deploy", "scaled", "scale", "monitoring",
    "latency", "throughput", "pipeline", "pipelines", "users", "serving",
    "real-time", "real time", "millions", "high-traffic", "uptime", "sla",
]

# ----- Pure-research detection
RESEARCH_KEYWORDS = [
    "research scientist", "phd researcher", "postdoc", "publication", "published",
    "paper", "academia", "university research", "thesis", "research fellow",
]

# ----- Like-to-have bonus groups
LIKE_TO_HAVE = {
    "fine_tuning": ["lora", "qlora", "peft", "fine-tun", "fine tun"],
    "learning_to_rank": ["lambdamart", "xgboost ranker", "learning to rank",
                          "neural rank", "ltr"],
    "hr_tech": ["recruitment", "talent matching", "hiring", "recruiter", "ats",
                "candidate matching", "hr-tech", "hr tech"],
    "distributed": ["inference optimization", "distributed serving", "distributed",
                    "quantization", "onnx", "triton", "tensorrt"],
    "open_source": ["open source", "open-source", "github", "contributor",
                    "maintainer", "oss"],
}

# ----- Domain penalty lexicons (CV / Speech / Robotics without IR evidence)
CV_KEYWORDS = ["yolo", "opencv", "segmentation", "object detection", "image classification", "cnn", "computer vision"]
SPEECH_KEYWORDS = ["asr", "wav2vec", "audio", "tts", "speech recognition", "voice"]
ROBOTICS_KEYWORDS = ["ros", "slam", "motion planning", "robotics", "lidar"]
IR_CORE_KEYWORDS = ["retrieval", "ranking", "recommendation", "search", "relevance", "evaluation", "embedding"]

print("Lexicons loaded.")

Lexicons loaded.


## 1. Data Loading &amp; Feature Engineering

Load `candidates.jsonl`, flatten the nested structures, and build a single
per-candidate feature table that downstream stages consume. All text is
lowercased and aggregated once here so the elimination/scoring stages are fast.

In [5]:
t0 = time.time()
raw = pd.read_json(DATA_PATH, lines=True)
print(f"Loaded {len(raw):,} candidates in {time.time()-t0:.1f}s")

# ---- small helpers -------------------------------------------------------
def _lower(x):
    return str(x).lower() if x is not None else ""

def any_kw(text, keywords):
    """True if any keyword is a substring of text (text already lowercased)."""
    return any(k in text for k in keywords)

def count_kw(text, keywords):
    """Number of distinct keywords found in text."""
    return sum(1 for k in keywords if k in text)

def parse_date(s):
    if not s or (isinstance(s, float) and np.isnan(s)):
        return pd.NaT
    return pd.to_datetime(s, errors="coerce")

Loaded 100,000 candidates in 13.0s


In [6]:
# ---------------- Build the per-candidate feature table -------------------
records = raw.to_dict("records")
rows = []

for rec in records:
    cid = rec.get("candidate_id")
    prof = rec.get("profile") or {}
    sig = rec.get("redrob_signals") or {}
    careers = rec.get("career_history") or []
    skills = rec.get("skills") or []

    # --- profile ---
    headline = _lower(prof.get("headline"))
    summary = _lower(prof.get("summary"))
    location = _lower(prof.get("location"))
    country = _lower(prof.get("country"))
    cur_title = _lower(prof.get("current_title"))
    yoe = prof.get("years_of_experience")
    yoe = float(yoe) if yoe is not None else np.nan

    # --- skills ---
    skill_names = [_lower(s.get("name")) for s in skills]
    skills_text = " ".join(skill_names)
    # relevant skill present (+ require positive duration when duration exists)
    has_relevant_skill = False
    for s in skills:
        nm = _lower(s.get("name"))
        if any(k in nm for k in RELEVANT_SKILLS):
            dur = s.get("duration_months")
            if dur is None or dur > 0:
                has_relevant_skill = True
                break
    # expert skills with ~0 duration (honeypot signal)
    expert_zero_dur = sum(
        1 for s in skills
        if _lower(s.get("proficiency")) == "expert" and (s.get("duration_months") or 0) <= 1
    )
    n_expert = sum(1 for s in skills if _lower(s.get("proficiency")) == "expert")

    # --- careers ---
    titles_past = [_lower(c.get("title")) for c in careers]
    titles_all = " | ".join([cur_title] + titles_past)
    descs = [_lower(c.get("description")) for c in careers]
    career_text = " ".join(descs)
    companies = [_lower(c.get("company")) for c in careers]
    durations = [float(c.get("duration_months") or 0) for c in careers]
    total_months = sum(durations)
    total_companies = len(set(companies))

    # job-hopping: fraction of jobs with tenure < 18 months
    short_jobs = sum(1 for d in durations if 0 < d < 18)
    short_switch_ratio = (short_jobs / len(durations)) if durations else 0.0

    # consulting exposure
    consulting_months = sum(
        d for c, d in zip(companies, durations) if any(k in c for k in CONSULTING_COMPANIES)
    )
    consulting_ratio = (consulting_months / total_months) if total_months > 0 else 0.0
    all_consulting = (
        len(companies) > 0 and all(any(k in c for k in CONSULTING_COMPANIES) for c in companies)
    )

    # title relevance
    title_relevant = any_kw(titles_all, RELEVANT_TITLE_KEYWORDS)
    # career-description evidence
    career_evidence = any_kw(career_text, CAREER_EVIDENCE_KEYWORDS)

    # entire-career-non-tech: every job non-tech, none tech-whitelisted, no AI evidence
    def _is_tech_title(t):
        return any(k in t for k in TECH_WHITELIST_TITLES)
    def _is_nontech_title(t):
        return any(k in t for k in NON_TECH_TITLE_KEYWORDS)
    all_titles = [t for t in ([cur_title] + titles_past) if t]
    non_tech_all = (
        len(all_titles) > 0
        and all((_is_nontech_title(t) and not _is_tech_title(t)) for t in all_titles)
        and not has_relevant_skill
        and not career_evidence
    )

    # research vs production (research months = tenure of clearly-research roles)
    research_months = 0.0
    for c, d in zip(careers, durations):
        ttl = _lower(c.get("title")) + " " + _lower(c.get("description"))
        if any(k in ttl for k in RESEARCH_KEYWORDS):
            research_months += d
    research_ratio = (research_months / total_months) if total_months > 0 else 0.0
    production_evidence = any_kw(career_text + " " + summary, PRODUCTION_KEYWORDS)

    # applied AI/ML years: sum tenure of roles showing AI/ML evidence
    applied_ai_months = 0.0
    for c, d in zip(careers, durations):
        blob = _lower(c.get("title")) + " " + _lower(c.get("description"))
        if any_kw(blob, RELEVANT_TITLE_KEYWORDS) or any_kw(blob, CAREER_EVIDENCE_KEYWORDS):
            applied_ai_months += d
    applied_ai_years = applied_ai_months / 12.0

    # full text used for semantic retrieval (titles + descriptions + skills + profile)
    full_text = " ".join([titles_all, career_text, skills_text, headline, summary]).strip()

    # --- signals ---
    rows.append({
        "candidate_id": cid,
        "headline": headline, "summary": summary,
        "location": location, "country": country,
        "current_title": cur_title, "years_of_experience": yoe,
        "current_company": _lower(prof.get("current_company")),
        "current_industry": _lower(prof.get("current_industry")),
        # skills / titles / text
        "skill_names": skill_names, "skills_text": skills_text,
        "titles_all": titles_all, "career_text": career_text,
        "full_text": full_text,
        # derived booleans / ratios
        "has_relevant_skill": has_relevant_skill,
        "title_relevant": title_relevant,
        "career_evidence": career_evidence,
        "n_expert": n_expert, "expert_zero_dur": expert_zero_dur,
        "total_months": total_months, "total_companies": total_companies,
        "short_switch_ratio": short_switch_ratio,
        "consulting_ratio": consulting_ratio, "all_consulting": all_consulting,
        "non_tech_all": non_tech_all,
        "research_ratio": research_ratio, "production_evidence": production_evidence,
        "applied_ai_years": applied_ai_years,
        # signals
        "open_to_work": bool(sig.get("open_to_work_flag")),
        "willing_to_relocate": bool(sig.get("willing_to_relocate")),
        "last_active_date": parse_date(sig.get("last_active_date")),
        "recruiter_response_rate": sig.get("recruiter_response_rate"),
        "interview_completion_rate": sig.get("interview_completion_rate"),
        "notice_period_days": sig.get("notice_period_days"),
        "github_activity_score": sig.get("github_activity_score"),
        "profile_completeness_score": sig.get("profile_completeness_score"),
    })

feat = pd.DataFrame(rows).set_index("candidate_id")
print(f"Feature table built: {feat.shape[0]:,} rows x {feat.shape[1]} cols in {time.time()-t0:.1f}s")

# Resolve reference date for recency
ref_date = pd.to_datetime(REFERENCE_DATE) if REFERENCE_DATE else feat["last_active_date"].max()
feat["days_since_active"] = (ref_date - feat["last_active_date"]).dt.days
print("Reference date for recency:", ref_date)
feat.head(3)

Feature table built: 100,000 rows x 35 cols in 117.8s
Reference date for recency: 2026-05-27 00:00:00


,headline,summary,location,country,current_title,years_of_experience,current_company,current_industry,skill_names,skills_text,titles_all,career_text,full_text,has_relevant_skill,title_relevant,career_evidence,n_expert,expert_zero_dur,total_months,total_companies,short_switch_ratio,consulting_ratio,all_consulting,non_tech_all,research_ratio,production_evidence,applied_ai_years,open_to_work,willing_to_relocate,last_active_date,recruiter_response_rate,interview_completion_rate,notice_period_days,github_activity_score,profile_completeness_score,days_since_active
candidate_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
CAND_0000001,"backend engineer | sql, spark, cloud",software / data professional with 6.9 years of...,toronto,canada,backend engineer,6.9,mindtree,it services,"[tailwind, nlp, image classification, fine-tun...",tailwind nlp image classification fine-tuning ...,backend engineer | backend engineer | analytic...,implemented streaming data pipelines on kafka ...,backend engineer | backend engineer | analytic...,True,False,True,0,0,82.0,2,0.00,0.329268,False,False,0.000000,True,6.833333,True,False,2026-05-20,0.34,0.71,60,9.2,86.9,7
CAND_0000002,operations manager | 12.5+ yrs experience,professional with 12.5+ years of experience. m...,"chennai, tamil nadu",india,operations manager,12.5,wipro,it services,"[project management, react, photoshop, typescr...",project management react photoshop typescript ...,operations manager | operations manager | oper...,customer support team lead at a saas product. ...,operations manager | operations manager | oper...,True,False,True,0,0,149.0,3,0.25,0.382550,False,False,0.362416,True,8.833333,True,False,2025-11-12,0.29,0.62,60,-1.0,78.7,196
CAND_0000003,customer support | 1.1+ yrs experience,professional with 1.1+ years of experience. i'...,austin,usa,customer support,1.1,tcs,it services,"[angular, seo, excel, accounting, kubernetes, ...",angular seo excel accounting kubernetes databr...,customer support | customer support,"business analyst at a consulting firm, working...",customer support | customer support business a...,False,False,True,0,0,13.0,1,1.00,1.000000,True,False,0.000000,False,1.083333,False,False,2026-03-21,0.46,0.86,150,-1.0,31.9,67


## Stage 1 — Direct Eliminations

A candidate is **eliminated** if they satisfy **ANY** hard reject rule below.
These are deliberately conservative: we only cut candidates extremely unlikely
to be selected per the JD. Everything else flows to scoring.

1. **Location** — not in preferred hubs AND not willing to relocate
2. **AI/ML evidence** — no relevant skill AND no relevant title AND no career evidence
3. **Extreme job-hopping** — (≥3 companies) and `short_switch_ratio > 0.7`
4. **Consulting-only** — every company is a consulting/services firm
5. **Entire-career non-tech** — every role non-tech, no AI evidence
6. **Pure research** *(optional)* — `research_ratio == 1` AND no production evidence

In [7]:
# --------------------------- Stage 1 filters ------------------------------
def in_preferred_location(loc):
    return any(h in loc for h in PREFERRED_LOCATIONS)

elim = pd.DataFrame(index=feat.index)

# 1. Location: reject if NOT in preferred hubs AND NOT willing to relocate
elim["loc_reject"] = (
    ~feat["location"].apply(in_preferred_location) & ~feat["willing_to_relocate"]
)

# 2. AI/ML evidence: reject if NO skill AND NO title AND NO career evidence
elim["no_ai_evidence"] = ~(
    feat["has_relevant_skill"] | feat["title_relevant"] | feat["career_evidence"]
)

# 3. Extreme job-hopping (only when >=3 companies)
elim["job_hopper"] = (feat["total_companies"] >= 3) & (feat["short_switch_ratio"] > 0.7)

# 4. Consulting-only career
elim["consulting_only"] = feat["all_consulting"]

# 5. Entire career non-tech
elim["non_tech"] = feat["non_tech_all"]

# 6. Pure research without production (optional)
elim["pure_research"] = (feat["research_ratio"] >= 0.999) & (~feat["production_evidence"])

elim["eliminated"] = elim.any(axis=1)

# ---- funnel report ----
print("=== STAGE 1: DIRECT ELIMINATIONS ===\n")
print(f"{'Total pool':35s}: {len(feat):>7,}")
for col, label in [
    ("loc_reject", "Rejected: location"),
    ("no_ai_evidence", "Rejected: no AI/ML evidence"),
    ("job_hopper", "Rejected: extreme job-hopping"),
    ("consulting_only", "Rejected: consulting-only"),
    ("non_tech", "Rejected: entire-career non-tech"),
    ("pure_research", "Rejected: pure research"),
]:
    print(f"  {label:33s}: {elim[col].sum():>7,}")

survivors = feat[~elim["eliminated"]].copy()
print(f"\n{'SURVIVORS -> Stage 2':35s}: {len(survivors):>7,}  "
      f"({len(survivors)/len(feat)*100:.1f}% of pool)")

=== STAGE 1: DIRECT ELIMINATIONS ===

Total pool                         : 100,000
  Rejected: location               :  53,437
  Rejected: no AI/ML evidence      :   3,941
  Rejected: extreme job-hopping    :     677
  Rejected: consulting-only        :   9,745
  Rejected: entire-career non-tech :   1,402
  Rejected: pure research          :       0

SURVIVORS -> Stage 2               :  40,242  (40.2% of pool)


## Stage 2 — Candidate Scoring

Every survivor receives sub-scores in **[0, 1]**, combined via the weighted
average in `WEIGHTS`, then multiplied by a CV/Robotics/Speech **penalty
multiplier**. Sub-scores:

- **A. Location** &nbsp; **B. Behavioral** &nbsp; **C. Experience** &nbsp; **D. JD Alignment (hybrid)**
- **E. Production Evidence** &nbsp; **F. Like-to-Have Bonus** &nbsp; **G. CV/Robotics/Speech Penalty**

In [8]:
# =================== A. LOCATION SCORE ====================================
def location_score(row):
    loc, country, relo = row["location"], row["country"], row["willing_to_relocate"]
    if any(h in loc for h in PREFERRED_PRIMARY):
        return 1.00
    if any(h in loc for h in PREFERRED_SECONDARY):
        return 0.95
    if "india" in country:
        return 0.80 if relo else 0.60
    # outside India
    return 0.40 if relo else 0.20

# =================== B. BEHAVIORAL SCORE ==================================
def _notice_score(days):
    if days is None or (isinstance(days, float) and np.isnan(days)):
        return 0.5
    if days <= 30:  return 1.00
    if days <= 60:  return 0.85
    if days <= 90:  return 0.60
    if days <= 120: return 0.30
    return 0.10

def _recency_score(days):
    # more recent activity -> higher. ~exponential decay over ~180 days.
    if days is None or (isinstance(days, float) and np.isnan(days)):
        return 0.4
    return float(np.clip(np.exp(-days / 120.0), 0.0, 1.0))

def behavioral_score(row):
    otw = 1.0 if row["open_to_work"] else 0.0
    rec = _recency_score(row["days_since_active"])
    rr = row["recruiter_response_rate"]
    rr = float(rr) if rr is not None and not pd.isna(rr) else 0.5
    ic = row["interview_completion_rate"]
    ic = float(ic) if ic is not None and not pd.isna(ic) else 0.5
    notice = _notice_score(row["notice_period_days"])
    # weighted blend (open-to-work + recency are very high priority)
    return (0.28 * otw + 0.27 * rec + 0.18 * rr + 0.12 * ic + 0.15 * notice)

# =================== C. EXPERIENCE SCORE ==================================
def _applied_ai_score(y):
    if y < 1:   return 0.00
    if y < 2:   return 0.20
    if y < 3:   return 0.40
    if y < 4:   return 0.70
    if y <= 5:  return 1.00
    if y <= 7:  return 0.90
    if y <= 10: return 0.75
    return 0.55

def _total_exp_score(y):
    if y is None or np.isnan(y): return 0.3
    if y < 3:   return 0.00
    if y < 5:   return 0.60
    if y <= 9:  return 1.00
    if y <= 15: return 0.80
    return 0.60

def experience_score(row):
    a = _applied_ai_score(row["applied_ai_years"])
    t = _total_exp_score(row["years_of_experience"])
    # applied-AI experience weighted higher than raw total experience
    return 0.6 * a + 0.4 * t

# =================== E. PRODUCTION EVIDENCE SCORE =========================
def production_score(row):
    blob = row["career_text"] + " " + row["summary"]
    hits = count_kw(blob, PRODUCTION_KEYWORDS)
    # saturate: 0 hits -> 0, 4+ distinct production cues -> 1.0
    return float(np.clip(hits / 4.0, 0.0, 1.0))

# =================== F. LIKE-TO-HAVE BONUS ================================
def like_to_have_score(row):
    blob = row["full_text"]
    groups_hit = sum(1 for kws in LIKE_TO_HAVE.values() if any_kw(blob, kws))
    return float(np.clip(groups_hit / len(LIKE_TO_HAVE), 0.0, 1.0))

# =================== G. CV/ROBOTICS/SPEECH PENALTY ========================
def domain_penalty(row):
    """Return (multiplier<=1.0, reason). Penalize off-domain specialists who
    show strong CV/Speech/Robotics focus but ~no IR/ranking evidence."""
    blob = row["full_text"]
    ir = count_kw(blob, IR_CORE_KEYWORDS)
    cv = count_kw(blob, CV_KEYWORDS)
    sp = count_kw(blob, SPEECH_KEYWORDS)
    ro = count_kw(blob, ROBOTICS_KEYWORDS)
    off_domain = max(cv, sp, ro)
    if ir == 0 and off_domain >= 2:
        return 0.65, "off-domain (CV/Speech/Robotics) focus, no IR evidence"
    if ir == 0 and off_domain == 1:
        return 0.85, "limited IR evidence, some off-domain focus"
    return 1.0, ""

print("Scoring functions A/B/C/E/F/G defined.")

Scoring functions A/B/C/E/F/G defined.


In [9]:
# =================== D. JD ALIGNMENT (HYBRID RETRIEVAL) ===================
# Positive JD representation: what an ideal candidate looks like.
JD_QUERY = (
    "senior ai engineer building production embeddings based retrieval and ranking "
    "systems for search and recommendation. strong python, vector databases such as "
    "pinecone weaviate qdrant milvus faiss elasticsearch opensearch. semantic search, "
    "information retrieval, learning to rank, bm25, re-ranking, relevance. fine-tuning "
    "large language models, llm, rag, transformers, sentence embeddings, hugging face. "
    "evaluation frameworks ndcg mrr map precision recall, offline and online a/b testing. "
    "deployed scalable low-latency machine learning pipelines to production serving users. "
    "mlops, model serving, monitoring, feature engineering."
)

_token_re = re.compile(r"[a-z0-9#+.]+")
def tokenize(text):
    return _token_re.findall(text)

surv_ids = survivors.index.tolist()
docs = survivors["full_text"].tolist()

def _minmax(arr):
    arr = np.asarray(arr, dtype=float)
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)

components = {}

# --- (1) TF-IDF cosine (always available via scikit-learn) ---
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000,
                            sublinear_tf=True, stop_words="english")
    doc_mat = tfidf.fit_transform(docs)
    q_mat = tfidf.transform([JD_QUERY])
    tfidf_sim = cosine_similarity(q_mat, doc_mat).ravel()
    components["tfidf"] = _minmax(tfidf_sim)
    print(f"TF-IDF cosine computed for {len(docs):,} docs.")
except Exception as e:
    print("TF-IDF unavailable:", e)

# --- (2) BM25 (rank_bm25) ---
try:
    from rank_bm25 import BM25Okapi
    tokenized = [tokenize(d) for d in docs]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = bm25.get_scores(tokenize(JD_QUERY))
    components["bm25"] = _minmax(bm25_scores)
    print("BM25 computed.")
except Exception as e:
    print("BM25 unavailable:", e)

# --- (3) Dense embeddings (sentence-transformers, optional) ---
if USE_DENSE_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
        st_model = SentenceTransformer(EMBED_MODEL)
        emb = st_model.encode(docs, batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)
        q_emb = st_model.encode([JD_QUERY], normalize_embeddings=True)
        dense_sim = (emb @ q_emb.T).ravel()
        components["dense"] = _minmax(dense_sim)
        print("Dense embeddings computed.")
    except Exception as e:
        print("Dense embeddings unavailable -> falling back to BM25+TF-IDF.", e)

# --- combine available components (weighted; dense favored when present) ---
comp_weights = {"dense": 0.5, "bm25": 0.3, "tfidf": 0.2}
present = {k: v for k, v in components.items() if k in comp_weights}
wsum = sum(comp_weights[k] for k in present)
jd_align = np.zeros(len(docs))
for k, v in present.items():
    jd_align += (comp_weights[k] / wsum) * v

survivors["score_jd_alignment"] = jd_align
print("\nJD alignment backends used:", list(present.keys()))
print(survivors["score_jd_alignment"].describe())

TF-IDF cosine computed for 40,242 docs.
BM25 computed.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\guran\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/629 [00:00<?, ?it/s]

Dense embeddings computed.

JD alignment backends used: ['tfidf', 'bm25', 'dense']
count    40242.000000
mean         0.235896
std          0.079016
min          0.025518
25%          0.181360
50%          0.230506
75%          0.274280
max          0.968686
Name: score_jd_alignment, dtype: float64


In [10]:
# ---- Blend JD-alignment with explicit retrieval/eval/LLM keyword evidence ----
def jd_keyword_evidence(row):
    blob = row["full_text"]
    r = min(count_kw(blob, RETRIEVAL_SIGNALS) / 4.0, 1.0)
    e = min(count_kw(blob, EVAL_SIGNALS) / 3.0, 1.0)
    l = min(count_kw(blob, LLM_SIGNALS) / 3.0, 1.0)
    # retrieval is the core competency for this role
    return 0.5 * r + 0.25 * e + 0.25 * l

survivors["jd_keyword_evidence"] = survivors.apply(jd_keyword_evidence, axis=1)
# 80% semantic similarity + 20% explicit signal evidence
survivors["score_jd_alignment"] = (
    0.8 * survivors["score_jd_alignment"] + 0.2 * survivors["jd_keyword_evidence"]
)

# ---- Compute remaining sub-scores ----
survivors["score_location"]   = survivors.apply(location_score, axis=1)
survivors["score_behavioral"] = survivors.apply(behavioral_score, axis=1)
survivors["score_experience"] = survivors.apply(experience_score, axis=1)
survivors["score_production"] = survivors.apply(production_score, axis=1)
survivors["score_like_to_have"] = survivors.apply(like_to_have_score, axis=1)

_pen = survivors.apply(domain_penalty, axis=1)
survivors["penalty_mult"] = _pen.apply(lambda x: x[0])
survivors["penalty_reason"] = _pen.apply(lambda x: x[1])

# ---- Weighted aggregation + penalty multiplier ----
base = (
    WEIGHTS["jd_alignment"] * survivors["score_jd_alignment"]
    + WEIGHTS["experience"]   * survivors["score_experience"]
    + WEIGHTS["behavioral"]   * survivors["score_behavioral"]
    + WEIGHTS["production"]    * survivors["score_production"]
    + WEIGHTS["location"]     * survivors["score_location"]
    + WEIGHTS["like_to_have"] * survivors["score_like_to_have"]
)
survivors["base_score"] = base
survivors["final_score"] = base * survivors["penalty_mult"]

print("Sub-scores + final score computed.")
score_cols = [c for c in survivors.columns if c.startswith("score_")] + ["penalty_mult", "final_score"]
survivors[score_cols].describe().round(3)

Sub-scores + final score computed.


,score_jd_alignment,score_location,score_behavioral,score_experience,score_production,score_like_to_have,penalty_mult,final_score
count,40242.000,40242.000,40242.000,40242.000,40242.000,40242.000,40242.000,40242.000
mean,0.245,0.828,0.467,0.653,0.361,0.226,0.925,0.402
std,0.091,0.197,0.165,0.272,0.291,0.112,0.088,0.098
min,0.037,0.400,0.103,0.000,0.000,0.000,0.650,0.103
25%,0.182,0.800,0.336,0.480,0.250,0.200,0.850,0.336
50%,0.233,0.950,0.435,0.770,0.250,0.200,1.000,0.402
75%,0.285,0.950,0.600,0.860,0.500,0.200,1.000,0.463
max,0.975,1.000,0.964,1.000,1.000,0.800,1.000,0.925


## Stage 3 — Honeypot Detection

Honeypots are impossible/contradictory profiles. We **reject** on strong
evidence and **penalize** when uncertain (configurable). Detectors:

- Expert skill claimed with ~zero usage duration (multiple times)
- Stated experience vs summed career tenure mismatch (impossible timelines)
- Title inflation / contradictory seniority vs tenure
- Implausible breadth of expert skills (keyword stuffing)

> Honeypot rate **> 10%** in the final top 100 = disqualification, so we screen the
> shortlist before locking it in.

In [11]:
# --------------------------- Honeypot scoring -----------------------------
# If uncertainty exists, we apply a penalty rather than a hard reject.
HONEYPOT_REJECT_THRESHOLD = 3   # cumulative suspicion points -> hard reject

def honeypot_flags(row):
    flags, points = [], 0

    # H1: multiple expert skills with ~zero duration -> strong
    if row["expert_zero_dur"] >= 3:
        flags.append("3+ expert skills with ~0 months usage"); points += 3
    elif row["expert_zero_dur"] == 2:
        flags.append("2 expert skills with ~0 months usage"); points += 2

    # H2: implausible number of expert skills (stuffing)
    if row["n_expert"] >= 10:
        flags.append(f"{row['n_expert']} expert skills (implausible breadth)"); points += 2

    # H3: stated experience vs summed career tenure mismatch
    yoe = row["years_of_experience"]
    career_years = row["total_months"] / 12.0
    if yoe is not None and not np.isnan(yoe) and career_years > 0:
        gap = abs(yoe - career_years)
        if gap > 8:
            flags.append(f"exp/tenure mismatch ({yoe:.0f}y stated vs {career_years:.0f}y career)"); points += 3
        elif gap > 5:
            flags.append(f"exp/tenure gap ({gap:.0f}y)"); points += 1

    # H4: title inflation — senior/lead/staff title with very little total tenure
    title = row["current_title"]
    if any(k in title for k in ["senior", "lead", "staff", "principal", "head"]) and career_years < 2:
        flags.append("senior-level title with <2y total tenure"); points += 2

    return points, "; ".join(flags)

_hp = survivors.apply(honeypot_flags, axis=1)
survivors["honeypot_points"] = _hp.apply(lambda x: x[0])
survivors["honeypot_reason"] = _hp.apply(lambda x: x[1])
survivors["is_honeypot"] = survivors["honeypot_points"] >= HONEYPOT_REJECT_THRESHOLD

# Soft penalty for uncertain (1-2 points): scale final score down a little
soft = survivors["honeypot_points"].between(1, HONEYPOT_REJECT_THRESHOLD - 1)
survivors.loc[soft, "final_score"] *= 0.90

print("=== STAGE 3: HONEYPOT DETECTION ===")
print(f"Hard-rejected honeypots:      {survivors['is_honeypot'].sum():,}")
print(f"Soft-penalized (uncertain):   {soft.sum():,}")

clean = survivors[~survivors["is_honeypot"]].copy()
print(f"Clean candidates remaining:   {len(clean):,}")

=== STAGE 3: HONEYPOT DETECTION ===
Hard-rejected honeypots:      21
Soft-penalized (uncertain):   12
Clean candidates remaining:   40,221


## Final Selection &amp; Reasoning Generation

Sort by `final_score` (descending) and take the **Top 100**. Then generate a
concise, candidate-specific explanation citing the strongest contributors, one
limitation, and a link to JD requirements.

In [12]:
# ----------------------------- Select Top 100 -----------------------------
top = clean.sort_values("final_score", ascending=False).head(TOP_N).copy()
top["rank"] = range(1, len(top) + 1)

# safety: confirm there is no honeypot leakage in the shortlist
hp_in_top = top["is_honeypot"].sum()
print(f"Top {len(top)} selected. Honeypots in shortlist: {hp_in_top} "
      f"({hp_in_top/len(top)*100:.1f}%) — must stay < 10%.")

Top 100 selected. Honeypots in shortlist: 0 (0.0%) — must stay < 10%.


In [13]:
# --------------------------- Reasoning generation -------------------------
# Human-readable phrases keyed by sub-score, weighted by importance.
STRENGTH_PHRASES = {
    "score_jd_alignment": lambda r: "strong semantic fit to the retrieval/ranking JD",
    "score_experience":   lambda r: f"{r['applied_ai_years']:.0f}y applied-AI experience"
                                     f" ({r['years_of_experience']:.0f}y total)",
    "score_production":   lambda r: "demonstrated production ML deployment",
    "score_behavioral":   lambda r: ("actively open to work and recently engaged"
                                      if r["open_to_work"] else "engaged recruiter signals"),
    "score_location":     lambda r: f"based in a preferred hub ({r['location'].title()})"
                                     if any(h in r['location'] for h in PREFERRED_LOCATIONS)
                                     else "geographically workable",
    "score_like_to_have": lambda r: "bonus depth (fine-tuning / LTR / OSS)",
}

def concern_phrase(r):
    # pick the weakest meaningful sub-score as the limitation
    subs = {
        "applied-AI depth": r["score_experience"],
        "production evidence": r["score_production"],
        "JD/domain alignment": r["score_jd_alignment"],
        "availability signals": r["score_behavioral"],
    }
    weakest = min(subs, key=subs.get)
    if r["penalty_reason"]:
        return f"concern: {r['penalty_reason']}"
    if r["honeypot_reason"]:
        return f"minor flag: {r['honeypot_reason']}"
    if subs[weakest] < 0.5:
        return f"limited {weakest}"
    if r["notice_period_days"] and r["notice_period_days"] > 60:
        return f"longer notice period (~{int(r['notice_period_days'])}d)"
    return "no major concerns; verify depth in interview"

# rotate sentence templates to avoid identical structure
TEMPLATES = [
    "Ranked #{rank}: {s1} and {s2}, with {s3}. {concern}.",
    "#{rank} — {s1}; also {s2}. Notably {s3}. {concern}.",
    "Selected (#{rank}) for {s1} plus {s2}. {s3.capitalize}. {concern}.",
    "#{rank}: combines {s1} with {s2} and {s3}. {concern}.",
]

def make_reasoning(cid, r, rank):
    # rank sub-scores by weighted contribution to pick top strengths
    contrib = {k: WEIGHTS[k.replace("score_", "")] * r[k] for k in STRENGTH_PHRASES}
    ordered = sorted(contrib, key=contrib.get, reverse=True)
    phrases = [STRENGTH_PHRASES[k](r) for k in ordered[:3]]
    tmpl = TEMPLATES[rank % len(TEMPLATES)]
    text = (tmpl
            .replace("{rank}", str(rank))
            .replace("{s1}", phrases[0])
            .replace("{s2}", phrases[1])
            .replace("{s3.capitalize}", phrases[2].capitalize())
            .replace("{s3}", phrases[2])
            .replace("{concern}", concern_phrase(r)))
    # collapse whitespace and tidy
    return re.sub(r"\s+", " ", text).strip()

top["reasoning"] = [make_reasoning(cid, row, row["rank"]) for cid, row in top.iterrows()]

# preview
for cid, row in top.head(5).iterrows():
    print(f"[{row['rank']:>3}] {cid}  score={row['final_score']:.4f}")
    print("     ", row["reasoning"], "\n")

[  1] CAND_0046525  score=0.9176
      #1 — strong semantic fit to the retrieval/ranking JD; also 6y applied-AI experience (6y total). Notably demonstrated production ML deployment. no major concerns; verify depth in interview. 

[  2] CAND_0018499  score=0.9038
      Selected (#2) for strong semantic fit to the retrieval/ranking JD plus 7y applied-AI experience (7y total). Demonstrated production ml deployment. no major concerns; verify depth in interview. 

[  3] CAND_0086022  score=0.8615
      #3: combines strong semantic fit to the retrieval/ranking JD with 5y applied-AI experience (5y total) and demonstrated production ML deployment. no major concerns; verify depth in interview. 

[  4] CAND_0096142  score=0.8381
      Ranked #4: strong semantic fit to the retrieval/ranking JD and 5y applied-AI experience (5y total), with demonstrated production ML deployment. longer notice period (~120d). 

[  5] CAND_0027691  score=0.8358
      #5 — strong semantic fit to the retrieval/ranking 

In [16]:
# ----------------------------- Write submission ---------------------------
submission = top.reset_index()[["candidate_id", "rank", "final_score", "reasoning"]].copy()
submission = submission.rename(columns={"final_score": "score"})

# round score for readability; ensure monotonically non-increasing by rank
submission["score"] = submission["score"].round(6)
submission = submission.sort_values("rank").reset_index(drop=True)
assert submission["score"].is_monotonic_decreasing or \
       (submission["score"].diff().dropna() <= 0).all(), "scores must be non-increasing"
assert len(submission) == TOP_N, f"expected {TOP_N} rows, got {len(submission)}"

submission.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV} with {len(submission)} rows.")
print(f"Total pipeline time: {time.time()-t0:.1f}s")
submission.head(10000)

Wrote submission.csv with 100 rows.
Total pipeline time: 2391.9s


,candidate_id,rank,score,reasoning
0,CAND_0046525,1,0.917591,#1 — strong semantic fit to the retrieval/rank...
1,CAND_0018499,2,0.903842,Selected (#2) for strong semantic fit to the r...
2,CAND_0086022,3,0.861497,#3: combines strong semantic fit to the retrie...
3,CAND_0096142,4,0.838105,Ranked #4: strong semantic fit to the retrieva...
4,CAND_0027691,5,0.835753,#5 — strong semantic fit to the retrieval/rank...
...,...,...,...,...
95,CAND_0087630,96,0.764717,Ranked #96: strong semantic fit to the retriev...
96,CAND_0070398,97,0.764148,#97 — strong semantic fit to the retrieval/ran...
97,CAND_0082086,98,0.764137,Selected (#98) for 6y applied-AI experience (6...
98,CAND_0074123,99,0.763971,#99: combines strong semantic fit to the retri...


In [15]:
# --------------------------- Sanity / QA report ---------------------------
print("=== TOP 100 QA SUMMARY ===")
print(f"Score range:        {top['final_score'].min():.4f} -> {top['final_score'].max():.4f}")
print(f"Honeypots in top:   {top['is_honeypot'].sum()} (DQ if >10)")
print(f"\nLocation mix (top 100):")
print(top['location'].apply(lambda l: next((h for h in PREFERRED_LOCATIONS if h in l), 'other/relocate')).value_counts().to_string())
print(f"\nYears of experience (top 100):")
print(top['years_of_experience'].describe().round(1).to_string())
print(f"\nApplied-AI years (top 100):")
print(top['applied_ai_years'].describe().round(1).to_string())
print(f"\nMean sub-scores (top 100):")
print(top[[c for c in top.columns if c.startswith('score_')]].mean().round(3).to_string())

=== TOP 100 QA SUMMARY ===
Score range:        0.7638 -> 0.9176
Honeypots in top:   0 (DQ if >10)

Location mix (top 100):
location
other/relocate    42
noida             18
pune              11
delhi              9
gurgaon            8
hyderabad          7
mumbai             5

Years of experience (top 100):
count    100.0
mean       6.2
std        1.0
min        3.0
25%        5.5
50%        6.2
75%        6.9
max        8.6

Applied-AI years (top 100):
count    100.0
mean       6.1
std        1.0
min        4.0
25%        5.4
50%        6.1
75%        6.8
max        8.4

Mean sub-scores (top 100):
score_jd_alignment    0.703
score_location        0.890
score_behavioral      0.791
score_experience      0.913
score_production      0.985
score_like_to_have    0.466


## Notes, Assumptions &amp; Tuning Knobs

**Key assumptions made (override in the config cells):**
- **Recency anchor** — the dataset is dated in the future, so `days_since_active` is measured from the *max* `last_active_date`. Set `REFERENCE_DATE` to fix this.
- **Applied-AI years** — estimated by summing tenure of roles whose title/description show AI/ML evidence (heuristic, not a labeled field).
- **Weights** in `WEIGHTS` are a sensible starting point; the spec calls for pairwise recruiter-style tuning (see clarifications).
- **Dense embeddings** require a one-time model download (network). During the timed run there is no network, so the model must already be cached, or set `USE_DENSE_EMBEDDINGS = False` (BM25 + TF-IDF only).

**Performance:** Stage 1 cuts the pool to a few thousand before any embedding work, so dense encoding stays well within the 5-min CPU budget. If it is ever tight, flip `USE_DENSE_EMBEDDINGS = False`.